In [ ]:
# Hücre 1: Ortam kurulumu
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = "/content/drive/MyDrive/SentinelAI"
os.chdir(BASE)
print(" Çalışma dizini:", os.getcwd())

Mounted at /content/drive
 Çalışma dizini: /content/drive/MyDrive/SentinelAI


In [ ]:
# Hücre 2 (REVİZE): Veriyi yükle + anomali skorlarını yeniden uygula
import pandas as pd
import joblib
import numpy as np

# 1. Veriyi yükle
df = pd.read_parquet(f"{BASE}/data/processed/logs.parquet")
print(f" Veri yüklendi: {len(df)} satır")

# 2. Kaydedilmiş modeli yükle
artifacts = joblib.load(f"{BASE}/models/isoforest.pkl")
model = artifacts["model"]
scaler = artifacts["scaler"]
features = artifacts["features"]
print(f" Model yüklendi. Özellikler: {features}")

# 3. Anomali tahminini yeniden uygula
X = df[features].values
X_scaled = scaler.transform(X)   #  fit_transform DEĞİL, sadece transform!

df["anomaly"] = model.predict(X_scaled)          # -1 = anomali, +1 = normal
df["anomaly_score"] = -model.score_samples(X_scaled)  # yüksek = daha anormal

print(f" Anomali tahmini uygulandı: {(df['anomaly']==-1).sum()} anomali")

# 4.  ÖNEMLİ: Bu sefer sütunları da kaydet!
df.to_parquet(f"{BASE}/data/processed/logs_with_anomaly.parquet", index=False)
print(f" Güncellenmiş veri kaydedildi: logs_with_anomaly.parquet")

# 5. Kontrol
gerekli = ["attempts", "is_failed", "is_night", "ip_log_count",
           "user_failed_count", "anomaly", "anomaly_score", "true_label"]
eksik = [c for c in gerekli if c not in df.columns]
print(f" Eksik: {eksik}" if eksik else " Tüm sütunlar mevcut")

 Veri yüklendi: 2100 satır
 Model yüklendi. Özellikler: ['attempts', 'is_failed', 'is_night', 'ip_log_count', 'user_failed_count']
 Anomali tahmini uygulandı: 104 anomali
 Güncellenmiş veri kaydedildi: logs_with_anomaly.parquet
 Tüm sütunlar mevcut


In [ ]:
# Hücre 2.5: Doğru dosyadan okuduğumuzu garanti et
df = pd.read_parquet(f"{BASE}/data/processed/logs_with_anomaly.parquet")
print(f" Doğru dosya yüklendi: {len(df)} satır")
print(f" Sütunlar: {list(df.columns)}")

 Doğru dosya yüklendi: 2100 satır
 Sütunlar: ['timestamp', 'ip', 'event', 'user', 'attempts', 'true_label', 'hour', 'is_night', 'is_failed', 'ip_log_count', 'user_failed_count', 'anomaly', 'anomaly_score']


In [ ]:
# Hücre 4: Kural tabanlı sınıflandırma
def classify_threat(row):
    """Bir log satırını sınıflandırır. (threat_type, mitre_id, reason) döner."""
    attempts   = row["attempts"]
    is_failed  = row["is_failed"]
    is_night   = row["is_night"]
    ip_count   = row["ip_log_count"]
    user_fail  = row["user_failed_count"]
    event      = row["event"]

    # --- Kural 1: Ağır brute-force ---
    if is_failed and attempts >= 20:
        return (
            "Brute Force",
            "T1110",
            f"Aynı oturumda {attempts} başarısız deneme. "
            f"IP toplam {ip_count} log üretmiş. Brute-force ile uyumlu."
        )

    # --- Kural 2: Orta seviye brute-force ---
    if is_failed and attempts >= 10:
        return (
            "Suspicious Failed Logins",
            "T1110.001",
            f"{attempts} başarısız deneme. Password guessing olabilir, izlenmeli."
        )

       # --- Kural 3: Gece başarılı giriş (İYİLEŞTİRİLMİŞ) ---
    if is_night and event == "Login Success":
        # Kullanıcının genel olarak gece girip girmediğine bakalım
        # Eğer o kullanıcı zaten hep gece giriyorsa (normal davranışıysa) alarm verme
        kullanici_gece_giris_sayisi = df[(df["user"] == row["user"]) & (df["is_night"] == 1) & (df["event"] == "Login Success")].shape[0]

        # Eğer bu kullanıcının toplam gece girişi çok azsa (örneğin 10'dan azsa) o zaman şüphelen
        # (Not: Sentetik veride herkesin 50-60 gece girişi olduğu için bu sayıyı 20 yapıyoruz, gerçekte 2-3 olur)
        if kullanici_gece_giris_sayisi < 20:
            return (
                "Suspicious Login Time",
                "T1078",
                f"Gece {int(row['hour']):02d}:00 civarında başarılı giriş. "
                f"Kullanıcının normal gece davranışı değil."
            )

    # --- Kural 4: Kullanıcıya yönelik yoğun fail ---
    if user_fail >= 20 and is_failed:
        return (
            "Targeted Credential Attack",
            "T1110.003",
            f"'{row['user']}' kullanıcısına {user_fail} başarısız giriş. "
            f"Hedefli saldırı göstergesi."
        )

    # --- Kural 5: Anomali ama kurala uymuyor ---
    if row["anomaly"] == -1:
        return (
            "Unknown Anomaly",
            "TBD",
            "Model anomali işaretledi fakat bilinen bir kurala uymuyor. "
            "Manuel inceleme gerekli."
        )

    # --- Kural 6: Normal ---
    return ("Normal", None, "Belirgin bir tehdit göstergesi yok.")


# Uygula
df[["threat_type", "mitre_id", "reason"]] = df.apply(
    lambda r: pd.Series(classify_threat(r)), axis=1
)

print(" Sınıflandırma tamamlandı\n")
print(" Tehdit dağılımı:")
print(df["threat_type"].value_counts().to_string())

print("\n MITRE dağılımı:")
print(df[df["mitre_id"].notna()]["mitre_id"].value_counts().to_string())

 Sınıflandırma tamamlandı

 Tehdit dağılımı:
threat_type
Normal             1988
Brute Force         100
Unknown Anomaly      12

 MITRE dağılımı:
mitre_id
T1110    100
TBD       12


In [ ]:
# Hücre 5: Her sınıftan örnek göster
print(" SINIF BAZLI ÖRNEKLER\n" + "="*70)

for threat in df["threat_type"].value_counts().index:
    if threat == "Normal":
        continue
    print(f"\n SINIF: {threat}  ({(df['threat_type']==threat).sum()} olay)")
    print("-"*70)
    subset = df[df["threat_type"] == threat].head(2)
    for _, row in subset.iterrows():
        print(f"   {row['timestamp']}")
        print(f"   IP: {row['ip']} |  Kullanıcı: {row['user']} |  Olay: {row['event']}")
        print(f"   Deneme: {row['attempts']} | Gece: {row['is_night']} | Fail: {row['is_failed']}")
        print(f"   MITRE: {row['mitre_id']}")
        print(f"   Gerekçe: {row['reason']}")
        print()

 SINIF BAZLI ÖRNEKLER

 SINIF: Brute Force  (100 olay)
----------------------------------------------------------------------
   2026-11-17 02:00:06
   IP: 192.168.1.45 |  Kullanıcı: admin |  Olay: Failed Login
   Deneme: 28 | Gece: 1 | Fail: 1
   MITRE: T1110
   Gerekçe: Aynı oturumda 28 başarısız deneme. IP toplam 100 log üretmiş. Brute-force ile uyumlu.

   2026-11-17 02:00:37
   IP: 192.168.1.45 |  Kullanıcı: admin |  Olay: Failed Login
   Deneme: 32 | Gece: 1 | Fail: 1
   MITRE: T1110
   Gerekçe: Aynı oturumda 32 başarısız deneme. IP toplam 100 log üretmiş. Brute-force ile uyumlu.


 SINIF: Unknown Anomaly  (12 olay)
----------------------------------------------------------------------
   2026-11-17 00:10:45
   IP: 192.168.1.28 |  Kullanıcı: admin |  Olay: Logout
   Deneme: 3 | Gece: 1 | Fail: 0
   MITRE: TBD
   Gerekçe: Model anomali işaretledi fakat bilinen bir kurala uymuyor. Manuel inceleme gerekli.

   2026-11-17 00:37:24
   IP: 192.168.1.28 |  Kullanıcı: admin |  Olay: Logi

In [ ]:
# Hücre 6: Kural motoru birim testleri
import pandas as pd

test_senaryolari = [
    {
        "ad": "Ağır brute-force",
        "row": {"attempts": 35, "is_failed": 1, "is_night": 1,
                "ip_log_count": 100, "user_failed_count": 100,
                "event": "Failed Login", "user": "admin", "hour": 2,
                "anomaly": -1},
        "beklenen": "Brute Force",
    },
    {
        "ad": "Orta seviye fail",
        "row": {"attempts": 12, "is_failed": 1, "is_night": 0,
                "ip_log_count": 50, "user_failed_count": 12,
                "event": "Failed Login", "user": "bob", "hour": 14,
                "anomaly": -1},
        "beklenen": "Suspicious Failed Logins",
    },
    {
        "ad": "Gece başarılı giriş",
        "row": {"attempts": 1, "is_failed": 0, "is_night": 1,
                "ip_log_count": 20, "user_failed_count": 0,
                "event": "Login Success", "user": "test_kullanici_123", "hour": 3,
                "anomaly": -1},
        "beklenen": "Suspicious Login Time",
    },
    {
        "ad": "Normal gündüz giriş",
        "row": {"attempts": 1, "is_failed": 0, "is_night": 0,
                "ip_log_count": 20, "user_failed_count": 0,
                "event": "Login Success", "user": "carol", "hour": 14,
                "anomaly": 1},
        "beklenen": "Normal",
    },
]

print(" Kural Motoru Testleri\n" + "="*60)
gecen = 0
for t in test_senaryolari:
    threat, mitre, reason = classify_threat(pd.Series(t["row"]))
    durum = "" if threat == t["beklenen"] else "❌"
    if threat == t["beklenen"]:
        gecen += 1
    print(f"{durum} {t['ad']:25} → {threat:30} (MITRE: {mitre})")
    if threat != t["beklenen"]:
        print(f"    Beklenen: {t['beklenen']} | Gelen: {threat}")

print("="*60)
print(f" Sonuç: {gecen}/{len(test_senaryolari)} test geçti")

 Kural Motoru Testleri
 Ağır brute-force          → Brute Force                    (MITRE: T1110)
 Orta seviye fail          → Suspicious Failed Logins       (MITRE: T1110.001)
 Gece başarılı giriş       → Suspicious Login Time          (MITRE: T1078)
 Normal gündüz giriş       → Normal                         (MITRE: None)
 Sonuç: 4/4 test geçti


In [ ]:
# Hücre 7: Kural motoru vs gerçek etiket
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_score, recall_score, f1_score
)

# "Saldırı mı?" → Normal değilse saldırı
df["pred_attack"] = (df["threat_type"] != "Normal").astype(int)

y_true = df["true_label"]
y_pred = df["pred_attack"]

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

print(" KARMAŞIKLIK MATRİSİ")
print(f"   TN={tn}  FP={fp}")
print(f"   FN={fn}  TP={tp}")
print()
print(" METRİKLER (Kural Tabanlı)")
print(f"   Precision : {precision_score(y_true, y_pred):.3f}")
print(f"   Recall    : {recall_score(y_true, y_pred):.3f}")
print(f"   F1        : {f1_score(y_true, y_pred):.3f}")
print(f"   FP oranı  : {fp/(fp+tn):.3f}")
print(f"   FN oranı  : {fn/(fn+tp):.3f}")
print()
print(" Sınıf bazlı rapor:")
print(classification_report(y_true, y_pred, target_names=["Normal", "Attack"], digits=3))

 KARMAŞIKLIK MATRİSİ
   TN=1988  FP=12
   FN=0  TP=100

 METRİKLER (Kural Tabanlı)
   Precision : 0.893
   Recall    : 1.000
   F1        : 0.943
   FP oranı  : 0.006
   FN oranı  : 0.000

 Sınıf bazlı rapor:
              precision    recall  f1-score   support

      Normal      1.000     0.994     0.997      2000
      Attack      0.893     1.000     0.943       100

    accuracy                          0.994      2100
   macro avg      0.946     0.997     0.970      2100
weighted avg      0.995     0.994     0.994      2100



In [ ]:
# Hücre 8: Sonuçları ve raporu kaydet
from sklearn.metrics import precision_score, recall_score, f1_score

# 1. Veriyi kaydet
out_path = f"{BASE}/data/processed/logs_classified.parquet"
df.to_parquet(out_path, index=False)
print(f" Sınıflandırılmış veri: {out_path}")

# 2. MITRE dağılımı
mitre_dist = df[df["mitre_id"].notna()]["mitre_id"].value_counts()

# 3. Markdown raporu
report = f"""# SentinelAI – Aşama 2: Threat Classification & MITRE Mapping

## Özet
- **Toplam log:** {len(df)}
- **Saldırı olarak işaretlenen:** {(df['threat_type'] != 'Normal').sum()}
- **Normal:** {(df['threat_type'] == 'Normal').sum()}

## Tehdit Dağılımı
{df['threat_type'].value_counts().to_markdown()}

## MITRE Dağılımı
{mitre_dist.to_markdown() if not mitre_dist.empty else 'MITRE kaydı bulunmamaktadır.'}
"""

# Raporu kaydet ve yazdır
report_path = f"{BASE}/reports/threat_report.md"
os.makedirs(os.path.dirname(report_path), exist_ok=True)
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report)

print(f" Rapor oluşturuldu ve kaydedildi: {report_path}\n")
print(report)

 Sınıflandırılmış veri: /content/drive/MyDrive/SentinelAI/data/processed/logs_classified.parquet
 Rapor oluşturuldu ve kaydedildi: /content/drive/MyDrive/SentinelAI/reports/threat_report.md

# SentinelAI – Aşama 2: Threat Classification & MITRE Mapping

## Özet
- **Toplam log:** 2100
- **Saldırı olarak işaretlenen:** 112
- **Normal:** 1988

## Tehdit Dağılımı
| threat_type     |   count |
|:----------------|--------:|
| Normal          |    1988 |
| Brute Force     |     100 |
| Unknown Anomaly |      12 |

## MITRE Dağılımı
| mitre_id   |   count |
|:-----------|--------:|
| T1110      |     100 |
| TBD        |      12 |

